# 04 · M1 — Detección de placas bolivianas + OCR
**IAM-811** · Laboratorio Final de Hito

1. **Detector**: YOLO11 (detección) de la clase `placa`, entrenado con un dataset de Roboflow
   (propio del notebook 01 o un dataset público de *license plates* combinado con fotos locales).
2. **OCR**: EasyOCR sobre el recorte (escala a 120 px de alto, CLAHE, Otsu) con lista blanca `A-Z 0-9`.
3. **Normalización boliviana**: mayúsculas, sin guiones ni espacios, formato `NNN(N)LLL` (vigente) o `LL(L)NNN(N)`
   (antiguo) y corrección por posición de `O↔0`, `I↔1`, `B↔8`, `S↔5`, `Z↔2`, `G↔6`.

**Contrato con `/inferencia`** (sección 6): archivo **`placas_yolo.pt`**, `task = detect`. Se entrena con
`single_cls=True`, así que aunque el dataset público tenga varias clases (`license-plate`, `plate`…) queda **una sola**
y el servidor la trata como `placa`. El OCR y la normalización los hace el servidor; aquí se evalúan.

In [ ]:
!nvidia-smi -L || echo "Sin GPU: Entorno de ejecución > Cambiar tipo de entorno > GPU T4"
!pip -q install -U ultralytics==8.4.218 roboflow==1.7.3 easyocr==1.7.2

## 1. Dataset de placas

In [ ]:
import os, re, glob, yaml, shutil, numpy as np, cv2, pandas as pd
FUENTE = 'roboflow'          # 'roboflow' o 'zip'
WORKSPACE, PROYECTO, VERSION = 'tu-workspace', 'peaje-placas', 1
NOMBRE_FINAL = 'placas_yolo.pt'       # nombre EXACTO que busca inferencia/app/config.py

def preparar_yaml(ruta_yaml):
    """Deja el data.yaml con rutas absolutas validas en Colab (Roboflow exporta '../train/images', 'valid/', etc.)."""
    d = yaml.safe_load(open(ruta_yaml))
    base = os.path.dirname(os.path.abspath(ruta_yaml))
    d['path'] = base
    for clave, candidatas in (('train', ['train/images', 'images/train']),
                              ('val', ['valid/images', 'val/images', 'images/val']),
                              ('test', ['test/images', 'images/test'])):
        encontrada = next((c for c in candidatas if os.path.isdir(os.path.join(base, c))), None)
        if encontrada:
            d[clave] = encontrada
        elif clave == 'test':
            d.pop('test', None)
        else:
            raise FileNotFoundError(f'No hay carpeta {clave} en {base}. En Roboflow genera la version con split train/valid.')
    yaml.safe_dump(d, open(ruta_yaml, 'w'), allow_unicode=True, sort_keys=False)
    return d

if FUENTE == 'roboflow':
    from google.colab import userdata
    from roboflow import Roboflow
    rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
    ds = rf.workspace(WORKSPACE).project(PROYECTO).version(VERSION).download('yolov11', location='/content/ds_placas')
    DATA_YAML = f'{ds.location}/data.yaml'
else:
    from google.colab import files
    for n in files.upload():
        !unzip -qo "$n" -d /content
    DATA_YAML = '/content/dataset_placas/data.yaml'
datos = preparar_yaml(DATA_YAML)
print(datos)

## 2. Entrenar el detector
`fliplr=0.0`: no se voltean las imágenes (los caracteres quedarían en espejo). `imgsz=640` = `IMGSZ_PROPIO` del servidor.

In [ ]:
from ultralytics import YOLO
YOLO('yolo11s.pt').train(data=DATA_YAML, epochs=80, imgsz=640, batch=16, patience=20, device=0,
                         fliplr=0.0, single_cls=True,
                         project='/content/runs/det', name='placas', exist_ok=True)
BEST = '/content/runs/det/placas/weights/best.pt'
m = YOLO(BEST).val(data=DATA_YAML, single_cls=True)
print(f'mAP50: {m.box.map50:.3f} | mAP50-95: {m.box.map:.3f}')

## 3. Normalización de placas de Bolivia (copia de `inferencia/app/normalizacion.py`)

In [ ]:
LETRA_A_DIGITO = {'O': '0', 'Q': '0', 'D': '0', 'I': '1', 'L': '1', 'T': '1', 'Z': '2', 'B': '8', 'S': '5', 'G': '6', 'A': '4'}
DIGITO_A_LETRA = {'0': 'O', '1': 'I', '2': 'Z', '8': 'B', '5': 'S', '6': 'G', '4': 'A', '7': 'T'}
RE_VIGENTE, RE_ANTIGUA = re.compile(r'^\d{3,4}[A-Z]{3}$'), re.compile(r'^[A-Z]{2,3}\d{3,4}$')

def limpiar(t):
    return re.sub(r'[^A-Z0-9]', '', (t or '').upper().replace('BOLIVIA', ''))

def normalizar(texto):
    t = limpiar(texto)
    if RE_VIGENTE.match(t) or RE_ANTIGUA.match(t):
        return t, True
    if len(t) in (6, 7):
        n = len(t) - 3
        c = ''.join(LETRA_A_DIGITO.get(ch, ch) if i < n and not ch.isdigit()
                    else DIGITO_A_LETRA.get(ch, ch) if i >= n and ch.isdigit() else ch for i, ch in enumerate(t))
        if RE_VIGENTE.match(c):
            return c, True
    return t, False

for e in ['1234-ABC', 'I234A8C', 'BOLIVIA 852 KLM', 'LPZ-123', 'HOLA']:
    print(f'{e!r:20} -> {normalizar(e)}')

## 4. OCR sobre los recortes del detector (igual que el servidor)

In [ ]:
import easyocr
lector = easyocr.Reader(['en'], gpu=True, verbose=False)
detector = YOLO(BEST)

def leer_placa(img):
    r = detector.predict(img, conf=0.25, imgsz=640, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return None, 0.0, None
    x1, y1, x2, y2 = r.boxes.xyxy.cpu().numpy()[r.boxes.conf.cpu().numpy().argmax()].astype(int)
    rec = img[max(0, y1 - 4):y2 + 4, max(0, x1 - 6):x2 + 6]
    f = max(1.0, 120 / max(1, rec.shape[0]))
    rec = cv2.resize(rec, None, fx=f, fy=f, interpolation=cv2.INTER_CUBIC)
    g = cv2.createCLAHE(2.0, (8, 8)).apply(cv2.cvtColor(rec, cv2.COLOR_BGR2GRAY))
    res = lector.readtext(g, allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-')
    if not res:
        return None, 0.0, (x1, y1, x2, y2)
    alto = max(abs(p[0][2][1] - p[0][0][1]) for p in res)
    trozos = sorted([p for p in res if abs(p[0][2][1] - p[0][0][1]) >= 0.5 * alto], key=lambda p: p[0][0][0])
    return ''.join(p[1] for p in trozos), float(np.mean([p[2] for p in trozos])), (x1, y1, x2, y2)

## 5. (Opcional) Evaluación end-to-end (detector + OCR + normalización)
Pon `EVALUAR_OCR = True` y sube imágenes (o un zip) y `placas_reales.csv` con columnas `archivo,placa` (placa sin guion).
Métricas: exactitud exacta de la placa y CER (tasa de error por carácter).

In [ ]:
EVALUAR_OCR = False

def cer(a, b):
    d = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, d[0] = d[0], i
        for j, cb in enumerate(b, 1):
            prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (ca != cb))
    return d[len(b)] / max(1, len(b))

if EVALUAR_OCR:
    from google.colab import files
    for n in files.upload():
        if n.endswith('.zip'):
            !unzip -qo "$n" -d /content/eval_placas
    filas = []
    for _, f in pd.read_csv('placas_reales.csv').iterrows():
        ruta = f['archivo'] if os.path.exists(f['archivo']) else os.path.join('/content/eval_placas', f['archivo'])
        img = cv2.imread(ruta)
        if img is None:
            continue
        texto, conf, _ = leer_placa(img)
        pred, valida = normalizar(texto)
        real = limpiar(f['placa'])
        filas.append({'archivo': f['archivo'], 'real': real, 'ocr': texto, 'normalizada': pred, 'valida': valida,
                      'conf': round(conf, 3), 'exacta': pred == real, 'CER': round(cer(pred, real), 3)})
    res = pd.DataFrame(filas)
    print(f"Exactitud: {res['exacta'].mean() * 100:.1f} % | CER medio: {res['CER'].mean():.3f}")
    display(res)

## 6. Verificar contrato y exportar `placas_yolo.pt`

In [ ]:
final = YOLO(BEST)
assert final.task == 'detect', f'task={final.task}: debe ser detect'
print('OK contrato M1 · task=detect · clases =', final.names)

shutil.copy(BEST, f'/content/{NOMBRE_FINAL}')
GUARDAR_DRIVE = True
if GUARDAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/peaje/pesos', exist_ok=True)
    shutil.copy(f'/content/{NOMBRE_FINAL}', f'/content/drive/MyDrive/peaje/pesos/{NOMBRE_FINAL}')
    print('Guardado en Drive: MyDrive/peaje/pesos/' + NOMBRE_FINAL)
from google.colab import files
files.download(f'/content/{NOMBRE_FINAL}')